# Exploración Visual de Arquetipos de Slack

Este notebook te permite:
- Visualizar arquetipos en espacio 3D usando PCA
- Explorar clusters de mensajes similares
- Identificar patrones y outliers
- Validar la calidad de la clasificación

In [17]:
# Instalar dependencias (ejecutar solo la primera vez)
# !pip install pandas numpy scikit-learn plotly psycopg2-binary python-dotenv sentence-transformers

In [18]:
import os
import pandas as pd
import numpy as np
import psycopg2
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go

# Cargar variables de entorno
load_dotenv()

print("✓ Librerías importadas")

✓ Librerías importadas


## 1. Cargar Datos desde PostgreSQL

In [19]:
# Conectar a la base de datos
DATABASE_URL = os.getenv('DATABASE_URL')
if not DATABASE_URL:
    raise ValueError('DATABASE_URL no encontrada en .env')

print('Conectando a PostgreSQL...')
conn = psycopg2.connect(DATABASE_URL)

# Query para obtener mensajes con JOIN a slack_users
# Filtrando: thread replies, reminders automáticos, y RoboCops
query = """
    SELECT
        sm.id,
        sm.text,
        sm.archetype,
        sm.category_role,
        sm.category_group,
        sm.datetime,
        su.name as user_name,
        sm.is_thread_reply
    FROM slack_messages sm
    LEFT JOIN slack_users su ON sm.user_id = su.id
    WHERE
        sm.text IS NOT NULL
        AND sm.text != ''
        AND sm.archetype IS NOT NULL
        AND sm.is_thread_reply = false
        AND sm.archetype NOT IN ('RoboCop', 'Reminder', 'Recordatorio Automático')
    ORDER BY sm.datetime DESC
    LIMIT 2000
"""

print('Cargando mensajes...')
print('Filtrando: thread replies, reminders automáticos, y RoboCops')
df = pd.read_sql_query(query, conn)
conn.close()

print(f'\n✓ Cargados {len(df)} mensajes (solo mensajes principales)')
print(f'✓ Arquetipos únicos: {df["archetype"].nunique()}')
print(f'\nDistribución de arquetipos:')
print(df['archetype'].value_counts())

Conectando a PostgreSQL...
Cargando mensajes...
Filtrando: thread replies, reminders automáticos, y RoboCops

✓ Cargados 495 mensajes (solo mensajes principales)
✓ Arquetipos únicos: 10

Distribución de arquetipos:
archetype
Reminder (Automático)                220
RoboCops (Bot)                        87
Sin Clasificar                        86
Tareas Operacionales Generales        29
Onboarding Merchants Kushki           28
Advertencias Bank Statements BICE     27
Reintentos Refund Disbursal            8
Gestión de Contracargos                7
Transferencias Estado-Security         2
Proceso Nómina Unired                  1
Name: count, dtype: int64


/var/folders/h2/22m5h8t51k1_m0spd6w6kfzm0000gn/T/ipykernel_18848/2379610322.py:35: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



## 2. Generar Embeddings

In [20]:
print('Cargando modelo de embeddings...')
model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

print('Generando embeddings (esto puede tardar unos minutos)...')
texts = df['text'].tolist()
embeddings = model.encode(texts, show_progress_bar=True)

print(f'\n✓ Embeddings generados: {embeddings.shape}')
print(f'  - {embeddings.shape[0]} mensajes')
print(f'  - {embeddings.shape[1]} dimensiones')

Cargando modelo de embeddings...


Generando embeddings (esto puede tardar unos minutos)...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]


✓ Embeddings generados: (495, 768)
  - 495 mensajes
  - 768 dimensiones


## 3. Reducción de Dimensionalidad con PCA

In [21]:
print('Aplicando PCA para reducir a 3 componentes principales...')
pca = PCA(n_components=3)
embeddings_3d = pca.fit_transform(embeddings)

# Agregar coordenadas al dataframe
df['pc1'] = embeddings_3d[:, 0]
df['pc2'] = embeddings_3d[:, 1]
df['pc3'] = embeddings_3d[:, 2]

# Varianza explicada
variance_explained = pca.explained_variance_ratio_
print(f'\n✓ Varianza explicada por cada componente:')
print(f'  PC1: {variance_explained[0]:.2%}')
print(f'  PC2: {variance_explained[1]:.2%}')
print(f'  PC3: {variance_explained[2]:.2%}')
print(f'  Total: {variance_explained.sum():.2%}')

Aplicando PCA para reducir a 3 componentes principales...

✓ Varianza explicada por cada componente:
  PC1: 15.64%
  PC2: 11.53%
  PC3: 8.68%
  Total: 35.86%


## 4. Visualización 3D Interactiva

In [22]:
# Crear texto de hover con preview del mensaje
df['text_preview'] = df['text'].str[:100].str.replace('\n', ' ')
df['hover_text'] = (
    '<b>' + df['archetype'] + '</b><br>' +
    'Usuario: ' + df['user_name'].fillna('Desconocido') + '<br>' +
    'Fecha: ' + df['datetime'].astype(str).str[:10] + '<br>' +
    '<i>' + df['text_preview'] + '...</i>'
)

# Crear figura 3D
fig = px.scatter_3d(
    df,
    x='pc1',
    y='pc2',
    z='pc3',
    color='archetype',
    hover_data={'hover_text': True, 'pc1': False, 'pc2': False, 'pc3': False},
    title=f'Visualización 3D de Arquetipos ({len(df)} mensajes)',
    labels={
        'pc1': f'PC1 ({variance_explained[0]:.1%})',
        'pc2': f'PC2 ({variance_explained[1]:.1%})',
        'pc3': f'PC3 ({variance_explained[2]:.1%})'
    },
    height=800,
    opacity=0.7
)

# Mejorar el layout
fig.update_traces(
    marker=dict(size=4, line=dict(width=0)),
    hovertemplate='%{customdata[0]}<extra></extra>'
)

fig.update_layout(
    scene=dict(
        xaxis=dict(backgroundcolor="rgb(240, 240, 240)"),
        yaxis=dict(backgroundcolor="rgb(240, 240, 240)"),
        zaxis=dict(backgroundcolor="rgb(240, 240, 240)"),
    ),
    font=dict(family="DM Sans, sans-serif", size=12),
)

fig.show()

print("\n💡 Tips:")
print("  - Arrastra para rotar la visualización")
print("  - Haz zoom con la rueda del mouse")
print("  - Pasa el cursor sobre los puntos para ver detalles")
print("  - Haz clic en los arquetipos de la leyenda para mostrar/ocultar")


💡 Tips:
  - Arrastra para rotar la visualización
  - Haz zoom con la rueda del mouse
  - Pasa el cursor sobre los puntos para ver detalles
  - Haz clic en los arquetipos de la leyenda para mostrar/ocultar


## 5. Análisis de Calidad de Clusters

In [23]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score

# Asignar etiquetas numéricas a los arquetipos
archetype_labels = df['archetype'].astype('category').cat.codes

# Calcular métricas de calidad
silhouette = silhouette_score(embeddings_3d, archetype_labels)
calinski = calinski_harabasz_score(embeddings_3d, archetype_labels)

print('📊 Métricas de Calidad de Clustering:\n')
print(f'Silhouette Score: {silhouette:.3f}')
print(f'  → Rango: [-1, 1]')
print(f'  → Más cerca de 1 = mejor separación entre clusters')
print()
print(f'Calinski-Harabasz Score: {calinski:.1f}')
print(f'  → Mayor valor = clusters más densos y separados')
print()

if silhouette > 0.5:
    print('✓ Excelente: Los clusters están bien definidos')
elif silhouette > 0.25:
    print('⚠️  Aceptable: Los clusters tienen cierta superposición')
else:
    print('❌ Pobre: Considerar revisar los arquetipos')

📊 Métricas de Calidad de Clustering:

Silhouette Score: -0.064
  → Rango: [-1, 1]
  → Más cerca de 1 = mejor separación entre clusters

Calinski-Harabasz Score: 116.0
  → Mayor valor = clusters más densos y separados

❌ Pobre: Considerar revisar los arquetipos


## 6. Explorar Arquetipos Específicos

In [24]:
# Listar todos los arquetipos disponibles
archetypes_list = df['archetype'].unique().tolist()
print(f'Arquetipos disponibles ({len(archetypes_list)}):\n')
for i, arch in enumerate(archetypes_list, 1):
    count = len(df[df['archetype'] == arch])
    print(f'{i}. {arch} ({count} mensajes)')

Arquetipos disponibles (10):

1. Reminder (Automático) (220 mensajes)
2. Sin Clasificar (86 mensajes)
3. Tareas Operacionales Generales (29 mensajes)
4. RoboCops (Bot) (87 mensajes)
5. Reintentos Refund Disbursal (8 mensajes)
6. Gestión de Contracargos (7 mensajes)
7. Transferencias Estado-Security (2 mensajes)
8. Onboarding Merchants Kushki (28 mensajes)
9. Advertencias Bank Statements BICE (27 mensajes)
10. Proceso Nómina Unired (1 mensajes)


In [25]:
# Seleccionar un arquetipo para explorar
# Cambiar este valor para explorar diferentes arquetipos
SELECTED_ARCHETYPE = archetypes_list[0]  # Cambiar índice aquí

print(f'📌 Explorando arquetipo: {SELECTED_ARCHETYPE}\n')

# Filtrar mensajes de este arquetipo
archetype_msgs = df[df['archetype'] == SELECTED_ARCHETYPE].copy()
print(f'Total de mensajes: {len(archetype_msgs)}\n')

# Mostrar ejemplos representativos
print('🔍 Ejemplos de mensajes:\n')
for idx, row in archetype_msgs.head(5).iterrows():
    text_clean = row['text'][:200].replace('\n', ' ')
    date = str(row['datetime'])[:10]
    user = row['user_name'] or 'Desconocido'
    print(f'[{date}] {user}')
    print(f'  "{text_clean}..."')
    print()

📌 Explorando arquetipo: Reminder (Automático)

Total de mensajes: 220

🔍 Ejemplos de mensajes:

[2026-02-03] Slackbot
  "Reminder: @S095YD2KQ4S descargar [nómina de cargos](http://admin.prd.fin/charge_batches), subirla al [banco](https://login.portalempresas.bancochile.cl/bancochile-web/empresa/login/index.html#/login) ..."

[2026-02-03] Slackbot
  "Reminder: @S095YD2KQ4S mover plata de Banco Estado al Security ([paso a paso](https://www.notion.so/fintoc/Subir-n-minas-Banco-Estado-Banco-Security-12e0ff41a1e9807f8dc1f11009203836))..."

[2026-02-03] Slackbot
  "Reminder: @S095YD2KQ4S sube el [universo](https://portalempresas.bancochile.cl/mibancochile-web/front/empresa/index.html#/portal/recaudacion-pac/universo) a [upload_contracts](http://admin.prd.fin/con..."

[2026-02-03] Slackbot
  "Reminder: @S08KGHVFHUN Hacer cierre contable de cuentas recaudadoras. [Acá](https://www.notion.so/fintoc/Cierres-contables-v-a-Ledger-2880ff41a1e98032876af54dac9a307f?source=copy_link) la guía...."

[202

## 7. Comparar Dos Arquetipos

In [26]:
# Seleccionar dos arquetipos para comparar
ARCH_A = archetypes_list[0]  # Cambiar índices aquí
ARCH_B = archetypes_list[1]

# Filtrar solo estos dos arquetipos
df_comparison = df[df['archetype'].isin([ARCH_A, ARCH_B])].copy()

# Crear visualización comparativa
fig_comp = px.scatter_3d(
    df_comparison,
    x='pc1',
    y='pc2',
    z='pc3',
    color='archetype',
    hover_data={'hover_text': True, 'pc1': False, 'pc2': False, 'pc3': False},
    title=f'Comparación: {ARCH_A} vs {ARCH_B}',
    labels={
        'pc1': f'PC1 ({variance_explained[0]:.1%})',
        'pc2': f'PC2 ({variance_explained[1]:.1%})',
        'pc3': f'PC3 ({variance_explained[2]:.1%})'
    },
    height=700,
    color_discrete_sequence=['#0045D7', '#DD0000']
)

fig_comp.update_traces(
    marker=dict(size=6, line=dict(width=0.5, color='white')),
    hovertemplate='%{customdata[0]}<extra></extra>'
)

fig_comp.show()

print(f'\n{ARCH_A}: {len(df_comparison[df_comparison["archetype"] == ARCH_A])} mensajes')
print(f'{ARCH_B}: {len(df_comparison[df_comparison["archetype"] == ARCH_B])} mensajes')


Reminder (Automático): 220 mensajes
Sin Clasificar: 86 mensajes


## 8. Exportar Datos para Análisis Externo

In [27]:
# Exportar coordenadas PCA con arquetipos
export_df = df[['id', 'text', 'archetype', 'datetime', 'user_name', 'pc1', 'pc2', 'pc3']].copy()
export_df.to_csv('archetypes_pca_3d.csv', index=False)

print('✓ Datos exportados a: archetypes_pca_3d.csv')
print('\nPuedes usar este archivo para:')
print('  - Análisis adicional en Python/R')
print('  - Visualización en Tableau/Power BI')
print('  - Compartir con el equipo')

✓ Datos exportados a: archetypes_pca_3d.csv

Puedes usar este archivo para:
  - Análisis adicional en Python/R
  - Visualización en Tableau/Power BI
  - Compartir con el equipo


## 9. Encontrar Mensajes Similares

In [28]:
from sklearn.metrics.pairwise import cosine_similarity

def find_similar_messages(query_text, top_k=5):
    """
    Encuentra los mensajes más similares a un texto de consulta.
    """
    # Generar embedding para la consulta
    query_embedding = model.encode([query_text])

    # Calcular similitud coseno
    similarities = cosine_similarity(query_embedding, embeddings)[0]

    # Obtener los top K más similares
    top_indices = similarities.argsort()[-top_k:][::-1]

    print(f'🔍 Mensajes más similares a: "{query_text}"\n')
    print('='*80)

    for rank, idx in enumerate(top_indices, 1):
        msg = df.iloc[idx]
        similarity = similarities[idx]

        print(f'\n#{rank} - Similitud: {similarity:.3f}')
        print(f'Arquetipo: {msg["archetype"]}')
        print(f'Fecha: {str(msg["datetime"])[:10]}')
        print(f'Usuario: {msg["user_name"] or "Desconocido"}')
        print(f'Texto: {msg["text"][:200].replace(chr(10), " ")}...')
        print('-'*80)

# Ejemplo de uso
find_similar_messages("problema con el pago", top_k=5)

🔍 Mensajes más similares a: "problema con el pago"


#1 - Similitud: 0.708
Arquetipo: Sin Clasificar
Fecha: 2026-01-20
Usuario: soledad
Texto: @S095YD2KQ4S me ayudan a ver si nos llegó este pago plisss. Y en el caso que si lo tengamos, no hay forma de adelantar la devolución?? :pray-intensifies: [PI](http://admin.prd.fin/payments/115722099)...
--------------------------------------------------------------------------------

#2 - Similitud: 0.695
Arquetipo: Sin Clasificar
Fecha: 2026-01-29
Usuario: cristobal.pinto
Texto: @S095YD2KQ4S tengo un usuario que sí tuvo descuento de dinero pero que el pago quedó `failed` ni siquiera quedó como IST, podemos revisar para hacer la devolución si corresponde • [Payment](http://adm...
--------------------------------------------------------------------------------

#3 - Similitud: 0.662
Arquetipo: Sin Clasificar
Fecha: 2026-02-03
Usuario: soledad
Texto: @S095YD2KQ4S me ayudas con la actualización del estado de este [pago](http://admin.prd.fin/payment

In [29]:
# Probar con tu propia consulta
CUSTOM_QUERY = "tu texto aquí"  # Cambiar esto
find_similar_messages(CUSTOM_QUERY, top_k=5)

🔍 Mensajes más similares a: "tu texto aquí"


#1 - Similitud: 0.684
Arquetipo: Sin Clasificar
Fecha: 2026-01-16
Usuario: pablo.vergara
Texto: :que-sucede:...
--------------------------------------------------------------------------------

#2 - Similitud: 0.627
Arquetipo: Sin Clasificar
Fecha: 2026-01-16
Usuario: pablo.vergara
Texto: :melt:...
--------------------------------------------------------------------------------

#3 - Similitud: 0.590
Arquetipo: Sin Clasificar
Fecha: 2026-01-30
Usuario: soledad
Texto: holii @mateo para enviar la información de los contracargos, creaste un wf? o aún no está listo? :dorime:...
--------------------------------------------------------------------------------

#4 - Similitud: 0.578
Arquetipo: Reminder (Automático)
Fecha: 2026-02-02
Usuario: soledad
Texto: :friendly-reminder: @mateo los 4 contracargos (uno fue levantado el viernes los otros 3 hoy) deben quedar listos hoy si o si plisssss :dorime: Gracias!...
---------------------------------------

## 🎯 Conclusiones y Próximos Pasos

Este notebook te permite:

1. **Validar arquetipos**: ¿Los clusters se ven separados en 3D?
2. **Identificar problemas**: ¿Hay arquetipos que se superponen?
3. **Descubrir patrones**: ¿Qué arquetipos son más frecuentes?
4. **Buscar similares**: Encuentra mensajes relacionados con un tema

### Acciones recomendadas:

- Si los clusters están muy mezclados → Revisar keywords en arquetipos manuales
- Si hay outliers → Crear nuevos arquetipos para capturarlos
- Si un arquetipo es muy grande → Considerar subdividirlo
- Si dos arquetipos se superponen → Fusionarlos o afinar keywords